In [1]:
training_data = [
    [("The", "DET"), ("dog", "NOUN"), ("barks", "VERB")],
    [("The", "DET"), ("cat", "NOUN"), ("meows", "VERB")],
    [("A", "DET"), ("dog", "NOUN"), ("runs", "VERB")],
    [("The", "DET"), ("dog", "NOUN"), ("runs", "VERB")],
    [("A", "DET"), ("cat", "NOUN"), ("sleeps", "VERB")],
    [("The", "DET"), ("big", "ADJ"), ("dog", "NOUN"), ("barks", "VERB")],
    [("A", "DET"), ("small", "ADJ"), ("cat", "NOUN"), ("meows", "VERB")],
    [("The", "DET"), ("dog", "NOUN"), ("sleeps", "VERB")],
    [("The", "DET"), ("cat", "NOUN"), ("runs", "VERB")],
    [("A", "DET"), ("big", "ADJ"), ("dog", "NOUN"), ("sleeps", "VERB")],
    [("The", "DET"), ("small", "ADJ"), ("cat", "NOUN"), ("runs", "VERB")],
    [("Dogs", "NOUN"), ("bark", "VERB")],
    [("Cats", "NOUN"), ("meow", "VERB")],
    [("The", "DET"), ("dog", "NOUN"), ("in", "ADP"), ("the", "DET"), ("house", "NOUN"), ("barks", "VERB")],
    [("A", "DET"), ("cat", "NOUN"), ("on", "ADP"), ("the", "DET"), ("mat", "NOUN"), ("sleeps", "VERB")],
]

In [2]:
import numpy as np
from collections import defaultdict, Counter
import math
import itertools

In [3]:
class HMMTagger:
    def __init__(self):
        self.vocab = set()
        self.tags = set()

        self.initial_counts = defaultdict(int)
        self.transition_counts = defaultdict(lambda: defaultdict(int)) # to avoid checking if exists
        self.emission_counts = defaultdict(lambda: defaultdict(int))
        self.tag_counts = defaultdict(int)

        self.total_size = 0
        self.vocab_size = 0
        self.tag_size = 0

    def get_initial_prob(self, tag):
        counts = self.initial_counts.get(tag, 0)
        return math.log(counts + 1) - math.log(self.total_size + self.tag_size) # same as / but with log it is -

    def get_transition_prob(self, prev_tag, curr_tag):
        count = self.transition_counts[prev_tag].get(curr_tag, 0)
        total_prev_counts = self.tag_counts.get(prev_tag, 0)
        return math.log(count + 1) - math.log(total_prev_counts + self.tag_size)

    def get_emission_prob(self, word, tag):
        count = self.emission_counts[tag].get(word, 0)
        total_tag_count = self.tag_counts.get(tag, 0)
        return math.log(count + 1) - math.log(total_tag_count + self.vocab_size)

    def train(self, training_data):
        self.total_size = len(training_data)

        for sentence in training_data:
            for i, (word, tag) in enumerate(sentence):
                self.vocab.add(word)
                self.tags.add(tag)
                self.tag_counts[tag] += 1
                self.emission_counts[tag][word] += 1
                if i == 0:
                    self.initial_counts[tag] += 1
                else:
                    prev_tag = sentence[i - 1][1]
                    self.transition_counts[prev_tag][tag] += 1
        self.vocab_size = len(self.vocab)
        self.tag_size = len(self.tags)
        print(f"Total sentences: {self.total_size}")
        print(f"Vocabulary size: {self.vocab_size}")
        print(f"Total unique tags: {self.tag_size} {self.tags}")

    def brute_force(self, sentence):
        all_tags = list(self.tags)
        num_words = len(sentence)

        all_sequences = itertools.product(all_tags, repeat=num_words)

        best_sequence = None
        max_prob = -float("inf")

        for sequence in all_sequences:
            current_prob = 0.0
            first_tag = sequence[0]
            first_word = sentence[0]
            current_prob += self.get_initial_prob(first_tag)
            current_prob += self.get_emission_prob(first_word, first_tag)

            for i in range(1, num_words):
                prev_tag = sequence[i - 1]
                current_tag = sequence[i]
                current_word = sentence[i]
                current_prob += self.get_transition_prob(prev_tag, current_tag)
                current_prob += self.get_emission_prob(current_word, current_tag)
            if current_prob > max_prob:
                max_prob = current_prob
                best_sequence = sequence

        return list(best_sequence)

test_sentences = [
    ["The", "dog", "barks"],
    ["A", "cat", "sleeps"],
    ["The", "big", "dog", "runs"],
    ["Dogs", "bark"],
    ["The", "small", "dog", "jumps"]
]

tagger = HMMTagger()
tagger.train(training_data)

Total sentences: 15
Vocabulary size: 19
Total unique tags: 5 {'ADJ', 'ADP', 'DET', 'NOUN', 'VERB'}


In [5]:
for sentence in test_sentences:
    tags = tagger.brute_force(sentence)
    print(f"Sentence: {sentence}")
    print(f"Tags:     {tags}\n")

Sentence: ['The', 'dog', 'barks']
Tags:     ['DET', 'NOUN', 'VERB']

Sentence: ['A', 'cat', 'sleeps']
Tags:     ['DET', 'NOUN', 'VERB']

Sentence: ['The', 'big', 'dog', 'runs']
Tags:     ['DET', 'ADJ', 'NOUN', 'VERB']

Sentence: ['Dogs', 'bark']
Tags:     ['NOUN', 'VERB']

Sentence: ['The', 'small', 'dog', 'jumps']
Tags:     ['DET', 'ADJ', 'NOUN', 'VERB']

